In [1]:
# install.packages(c("ggplot2", "ggpubr", "dplyr", "stringr", "tibble", "glue", "data.table"))

In [2]:
suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(ggpubr)
  library(glue)
  library(stringr)
  library(tibble)
  library(data.table)
})

options(warn = -1)
source("../../../manuscript-figures/utilities/functions/label-and-color-maps.R")

# tp_color_map <- c(
#   "PreTx"    = "#8B0000",
#   "PI2C"     = "#C20272",
#   "EI"       = "#7502C2",
#   "ASCT60d"  = "#0021B0",
#   "ASCT90d"  = "#2346DE",
#   "ASCT1y"   = "#038AFF",
#   "ASCT2y"   = "#008080",
#   "Healthy"  = "#00470D"
# )

In [3]:
match_pathways <- function(user_pathways, gsva_all, fuzzy = FALSE,
                           prefer = c("HALLMARK", "REACTOME", "ENCODE", "KEGG")) {
  norm <- function(x) trimws(toupper(gsub("_", " ", sub("^[A-Z0-9]+_", "", x))))
  all_pw   <- unique(gsva_all$pathway)
  norm_all <- norm(all_pw)

  # Pick the single best pathway from a vector of candidates,
  # using `prefer` order on the source prefix; ties / unknowns keep input order.
  pick_one <- function(hits) {
    if (length(hits) <= 1) return(hits)
    src  <- sub("_.*$", "", hits)
    rank <- match(src, prefer)
    rank[is.na(rank)] <- length(prefer) + 1L
    hits[order(rank)][1]
  }

  out <- lapply(norm(user_pathways), function(u) {
    hits <- if (fuzzy) all_pw[grepl(u, norm_all, fixed = TRUE)]
            else       all_pw[norm_all == u]
    pick_one(hits)
  })
  names(out) <- user_pathways

  miss <- names(out)[lengths(out) == 0]
  if (length(miss)) message("No match: ", paste(miss, collapse = "; "))

  unname(unlist(out))
}

In [4]:
plot_gsva_violins <- function(celltype, pathways_to_plot, tissue, save = TRUE) {
  # --- Prepare GSVA scores for the requested cell type & pathways ---
  scores_long <- gsva_all %>%
    filter(celltype == .env$celltype, pathway %in% pathways_to_plot) %>%
    mutate(
      # Enforce the chronological visit ordering on the x-axis
      label.visitDetails = factor(label.visitDetails, levels = tp_order),
      # Build a clean facet label: drop the collection prefix (e.g. "HALLMARK_")
      # and convert remaining underscores to spaces, in upper case
      pathway_label = stringr::str_to_upper(
        stringr::str_replace(pathway, "^[^_]+_", "") %>%
          stringr::str_replace_all("_", " ")
      )
    ) %>%
    filter(!is.na(label.visitDetails))

  # Bail out early if nothing matched the filters
  if (nrow(scores_long) == 0) {
    message("No data for: ", celltype)
    return(NULL)
  }

  # Build significance brackets from precomputed pairwise stats
  brackets <- stats_all %>%
    filter(
      celltype == .env$celltype,
      pathway %in% pathways_to_plot,
      p_signif != "ns"
    ) %>%
    # Keep only the comparisons we actually want to display
    semi_join(
      tibble::tibble(
        group1 = sapply(stat_comparisons, `[`, 1),
        group2 = sapply(stat_comparisons, `[`, 2)
      ),
      by = c("group1", "group2")
    ) %>%
    mutate(
      pathway_label = stringr::str_to_upper(
        stringr::str_replace(pathway, "^[^_]+_", "") %>%
          stringr::str_replace_all("_", " ")
      )
    )

  # Compute a y-position for each bracket so they stack neatly above each facet
  if (nrow(brackets) > 0) {
    rng <- scores_long %>%
      group_by(pathway_label) %>%
      summarise(
        ymax = max(gsva_score, na.rm = TRUE),
        ymin = min(gsva_score, na.rm = TRUE),
        .groups = "drop"
      ) %>%
      mutate(step = 0.10 * (ymax - ymin)) # vertical spacing between brackets

    brackets <- brackets %>%
      left_join(rng, by = "pathway_label") %>%
      group_by(pathway_label) %>%
      # Order brackets by the chronological position of group2
      arrange(match(group2, tp_order), .by_group = TRUE) %>%
      mutate(y.position = ymax + step * row_number()) %>%
      ungroup()
  }

  # Figure dimensions: 2 rows of facets, width scales with ncol
  n_pathways <- length(pathways_to_plot)
  n_rows <- 2
  ncol <- ceiling(n_pathways / n_rows)
  width <- ncol * 4
  height <- n_rows * 4

  line_data <- scores_long %>%
    filter(label.visitDetails != "Healthy") %>%
    group_by(subject.subjectGuid, pathway_label) %>%
    filter(dplyr::n() >= 2) %>%
    ungroup()
  # Assemble the plot
  p <- ggplot(scores_long, aes(
    x = label.visitDetails, y = gsva_score,
    fill = label.visitDetails, color = label.visitDetails
  )) +
    # Violin distribution per visit
    geom_violin(alpha = 0.3, linewidth = 0.5, trim = TRUE) +
    # Connect repeat measurements from the same subject (excluding Healthy controls)
    geom_line(
      data = line_data,
      aes(group = subject.subjectGuid),
      color = "black", linewidth = 0.3, alpha = 0.5
    ) +
    # Individual sample points
    geom_jitter(
      width = 0.1, size = 2, shape = 21, alpha = 0.7, color = "black"
    ) +
    scale_fill_manual(values = tp_color_map) +
    scale_color_manual(values = tp_color_map) +
    # One facet per pathway; wrap long titles
    facet_wrap(
      ~pathway_label,
      scales   = "free_y",
      ncol     = ncol,
      labeller = labeller(pathway_label = label_wrap_gen(30))
    ) +
    theme_classic(base_size = 14) +
    theme(
      axis.text.x      = element_text(angle = 45, hjust = 1, size = 12),
      axis.text.y      = element_text(size = 12),
      axis.title.y     = element_text(size = 13),
      strip.text       = element_text(face = "bold", size = 12),
      strip.background = element_blank(),
      plot.title       = element_text(face = "bold", size = 14, hjust = 0.5),
      legend.position  = "none"
    ) +
    labs(
      x     = NULL,
      y     = "GSVA Score",
      title = paste0(toupper(tissue), " - ", celltype)
    )

  # Overlay significance brackets when we have any to show
  if (nrow(brackets) > 0) {
    p <- p + ggpubr::stat_pvalue_manual(
      brackets,
      label        = "p_signif",
      xmin         = "group1",
      xmax         = "group2",
      y.position   = "y.position",
      tip.length   = 0.01,
      bracket.size = 0.3,
      size         = 4,
      fontface     = "bold"
    )
  }

  # --- Optionally write the figure to disk ---
  if (save) {
    out_file <- file.path(
      plot_dir,
      glue("gsva_{tissue}_{gsub(' ', '_', celltype)}.pdf")
    )
    ggsave(out_file, plot = p, width = width, height = height, device = cairo_pdf)
    message("Saved: ", out_file)
  }

  invisible(p)
}

## Revision Figure 3C - BMMC

In [5]:
plot_dir <- "../../../data/rna/gsva/results/gsva_plots/fig_3c"
out_dir <- "../../../data/rna/gsva/results/gsva_results"
dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)

tp_order <- c("PreTx", "EI", "ASCT1y", "Healthy")
stat_comparisons <- list(c("Healthy", "PreTx"), c("Healthy", "EI"), c("Healthy", "ASCT1y"))

tissue <- "bmmc" 
gsva_all <- fread(file.path(out_dir, sprintf("gsva_scores_%s.csv", tissue)))
stats_all <- fread(file.path(out_dir, sprintf("gsva_stats_%s_adjusted.csv", tissue)))

celltypes_to_plot <- c(
  "CMP Core",
  "HSPC Multi",
  "Prog B Pre",
  "CD16 Mono Core",
  "CD14 Mono Core",
  "pDC", "cDC2 Core",
  "Treg",
  "Naive B Core",
  "CD8 T Naive Core",
  "CD4 T Naive Core",
  "CD4 T EM1",
  "CD8 T EM1",
  "CD56dim NK GZMK+",
  "CD56br NK",
  "Mem B Core"
)

pathways_to_plot <- match_pathways(c(
  "CEBPD ENCODE",
  "CYTOKINE SIGNALING IN IMMUNE SYSTEM",
  "EPSTEIN-BARR VIRUS INFECTION",
  "HYPOXIA",
  "INTERFERON ALPHA RESPONSE",
  "INTERFERON GAMMA RESPONSE",
  "OXIDATIVE PHOSPHORYLATION",
  "TCR SIGNALING",
  "TGF-BETA SIGNALING",
  "TNF-ALPHA SIGNALING VIA NF-KB"
), gsva_all)

In [ ]:
for (ct in celltypes_to_plot) {
  plot_gsva_violins(ct, pathways_to_plot, tissue)
}

Saved: results/gsva_plots/fig_3c/gsva_bmmc_CMP_Core.pdf

Saved: results/gsva_plots/fig_3c/gsva_bmmc_HSPC_Multi.pdf

Saved: results/gsva_plots/fig_3c/gsva_bmmc_Prog_B_Pre.pdf

Saved: results/gsva_plots/fig_3c/gsva_bmmc_CD16_Mono_Core.pdf

Saved: results/gsva_plots/fig_3c/gsva_bmmc_CD14_Mono_Core.pdf

Saved: results/gsva_plots/fig_3c/gsva_bmmc_pDC.pdf

Saved: results/gsva_plots/fig_3c/gsva_bmmc_cDC2_Core.pdf

Saved: results/gsva_plots/fig_3c/gsva_bmmc_Treg.pdf

Saved: results/gsva_plots/fig_3c/gsva_bmmc_Naive_B_Core.pdf

Saved: results/gsva_plots/fig_3c/gsva_bmmc_CD8_T_Naive_Core.pdf



## Revision Figure 3D - PBMC

In [ ]:
plot_dir <- "../../../data/rna/gsva/results/gsva_plots/fig_3d"
out_dir <- "../../../data/rna/gsva/results/gsva_results"
dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)

tp_order <- c("PreTx", "EI", "ASCT1y", "Healthy")
stat_comparisons <- list(c("Healthy", "PreTx"), c("Healthy", "EI"), c("Healthy", "ASCT1y"))

tissue <- "pbmc" 
gsva_all <- fread(file.path(out_dir, sprintf("gsva_scores_%s.csv", tissue)))
stats_all <- fread(file.path(out_dir, sprintf("gsva_stats_%s_adjusted.csv", tissue)))

celltypes_to_plot <- c(
  "Treg Naive",
  "CD16 Mono Core",
  "CD14 Mono Core",
  "pDC",
  "CD8 MAIT",
  "CD4 MAIT",
  "Trans B",
  "Naive B ISG+",
  "Naive B Core",
  "CD8 T Naive Core",
  "CD4 T Naive SOX4+",
  "CD4 T Naive ISG+",
  "CD4 T Naive Core",
  "CD4 T Mem KLRF1- GZMB+",
  "CD4 T Mem ISG+",
  "CD4 T EM GZMB- CD27-",
  "CD4 T EM GZMB- CD27+",
  "gdT Vd2 GZMK+",
  "gdT Vd2 GZMB+",
  "gdT Vd1 Eff KLRF1+",
  "Prolif T",
  "Prolif NK",
  "CD8 T Mem ISG+",
  "CD8 T EM KLRF1- GZMB+",
  "CD8 T EM KLRF1+ GZMB+",
  "CD8 T EM GZMK- CD27+",
  "CD8 T EM GZMK+ CD27+",
  "NK Adaptive",
  "CD56dim NK ISG+",
  "CD56dim NK GZMK+",
  "CD56dim NK GZMK-",
  "CD56br NK",
  "Mem B Type2",
  "Mem B Early",
  "Mem B Core",
  "Mem B CD95",
  "Eff B CD27-",
  "Eff B CD27+"
)

pathways_to_plot <- match_pathways(c(
  "CEBPD ENCODE",
  "CYTOKINE SIGNALING IN IMMUNE SYSTEM",
  "EPSTEIN-BARR VIRUS INFECTION",
  "HYPOXIA",
  "INTERFERON ALPHA RESPONSE",
  "INTERFERON GAMMA RESPONSE",
  "OXIDATIVE PHOSPHORYLATION",
  "TCR SIGNALING",
  "TGF-BETA SIGNALING",
  "TNF-ALPHA SIGNALING VIA NF-KB"
), gsva_all)

In [8]:
for (ct in celltypes_to_plot) {
  plot_gsva_violins(ct, pathways_to_plot, tissue)
}

Saved: results/gsva_plots/fig_3d/gsva_pbmc_CD8_T_EM_GZMK+_CD27+.pdf

Saved: results/gsva_plots/fig_3d/gsva_pbmc_NK_Adaptive.pdf

Saved: results/gsva_plots/fig_3d/gsva_pbmc_CD56dim_NK_ISG+.pdf

Saved: results/gsva_plots/fig_3d/gsva_pbmc_CD56dim_NK_GZMK+.pdf

Saved: results/gsva_plots/fig_3d/gsva_pbmc_CD56dim_NK_GZMK-.pdf

Saved: results/gsva_plots/fig_3d/gsva_pbmc_CD56br_NK.pdf

Saved: results/gsva_plots/fig_3d/gsva_pbmc_Mem_B_Type2.pdf

Saved: results/gsva_plots/fig_3d/gsva_pbmc_Mem_B_Early.pdf

Saved: results/gsva_plots/fig_3d/gsva_pbmc_Mem_B_Core.pdf

Saved: results/gsva_plots/fig_3d/gsva_pbmc_Mem_B_CD95.pdf

Saved: results/gsva_plots/fig_3d/gsva_pbmc_Eff_B_CD27-.pdf

Saved: results/gsva_plots/fig_3d/gsva_pbmc_Eff_B_CD27+.pdf



## Revision Figure 4B - BMMC

In [9]:
plot_dir <- "../../../data/rna/gsva/results/gsva_plots/fig_4b"
out_dir <- "../../../data/rna/gsva/results/gsva_results"
dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)

tp_order <- c("PreTx", "EI", "ASCT1y", "Healthy")
stat_comparisons <- list(c("Healthy", "PreTx"), c("Healthy", "EI"), c("Healthy", "ASCT1y"))

tissue <- "bmmc" 
gsva_all <- fread(file.path(out_dir, sprintf("gsva_scores_%s.csv", tissue)))
stats_all <- fread(file.path(out_dir, sprintf("gsva_stats_%s_adjusted.csv", tissue)))

celltypes_to_plot <- c(
  "HSPC Stem",
  "HSPC Multi",
  "LMPP",
  "CLP",
  "CMP Core",
  "CMP Gran",
  "MEP",
  "Pre Prog Ery",
  "Prog Ery",
  "Prog Ery Prolif",
  "Pre Mono Core",
  "Pre Mono Prolif",
  "Prog B Pre",
  "Prog B",
  "Pre B Heavy",
  "Pre B Light",
  "Pre B Prolif",
  "Prog DC cDC",
  "Prog DC pDC"
)

pathways_to_plot <- match_pathways(c(
  "IRF3 ENCODE",
  "INTERFERON GAMMA RESPONSE",
  "INTERFERON ALPHA RESPONSE",
  "EPSTEIN-BARR VIRUS INFECTION",
  "TGF-BETA SIGNALING",
  "TNF-ALPHA SIGNALING VIA NF-KB",
  "HYPOXIA",
  "OXIDATIVE PHOSPHORYLATION",
  "FOXM1 ENCODE"
), gsva_all)

In [10]:
for (ct in celltypes_to_plot) {
  plot_gsva_violins(ct, pathways_to_plot, tissue)
}

Saved: results/gsva_plots/fig_4b/gsva_bmmc_HSPC_Stem.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_HSPC_Multi.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_LMPP.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_CLP.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_CMP_Core.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_CMP_Gran.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_MEP.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_Pre_Prog_Ery.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_Prog_Ery.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_Prog_Ery_Prolif.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_Pre_Mono_Core.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_Pre_Mono_Prolif.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_Prog_B_Pre.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_Prog_B.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_Pre_B_Heavy.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_Pre_B_Light.pdf

Saved: results/gsva_plots/fig_4b/gsva_bmmc_Pre_B_Prolif.pdf

Saved: re

## Revision Figure 5E - BMMC

In [11]:
plot_dir <- "../../../data/rna/gsva/results/gsva_plots/fig_5e"
out_dir <- "../../../data/rna/gsva/results/gsva_results"
dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)

tp_order <- c("PreTx", "EI", "ASCT90d", "ASCT1y", "ASCT2y", "Healthy")
stat_comparisons <- list(c("Healthy", "PreTx"), c("Healthy", "EI"), c("Healthy", "ASCT90d"), c("Healthy", "ASCT1y"), c("Healthy", "ASCT2y"))

tissue <- "bmmc" 
gsva_all <- fread(file.path(out_dir, sprintf("gsva_scores_%s.csv", tissue)))
stats_all <- fread(file.path(out_dir, sprintf("gsva_stats_%s_adjusted.csv", tissue)))

celltypes_to_plot <- c(
  "CD8 T Naive Core",
  "CD4 T Naive Core", 
  "CD4 T EM1", 
  "CD4 T EM2", 
  "gdT", 
  "CD8 T Tissue Res",
  "CD8 T EM2", 
  "CD8 T EM1", 
  "NK Tissue Res", 
  "NK Effector", 
  "CD56dim NK GZMK-",
  "CD56dim NK GZMK+", 
  "CD56br NK"
)

pathways_to_plot <- match_pathways(c(
  "EPSTEIN-BARR VIRUS INFECTION",
  "HYPOXIA",
  "INTERFERON ALPHA RESPONSE",
  "INTERFERON GAMMA RESPONSE",
  "OXIDATIVE PHOSPHORYLATION",
  "TCR SIGNALING",
  "TGF-BETA SIGNALING",
  "TNF-ALPHA SIGNALING VIA NF-KB"
), gsva_all)

In [12]:
for (ct in celltypes_to_plot) {
  plot_gsva_violins(ct, pathways_to_plot, tissue)
}

Saved: results/gsva_plots/fig_5e/gsva_bmmc_CD8_T_Naive_Core.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_CD4_T_Naive_Core.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_CD4_T_EM1.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_CD4_T_EM2.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_gdT.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_CD8_T_Tissue_Res.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_CD8_T_EM2.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_CD8_T_EM1.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_NK_Tissue_Res.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_NK_Effector.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_CD56dim_NK_GZMK-.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_CD56dim_NK_GZMK+.pdf

Saved: results/gsva_plots/fig_5e/gsva_bmmc_CD56br_NK.pdf



## Revision Figure 5E - PBMC

In [13]:
plot_dir <- "../../../data/rna/gsva/results/gsva_plots/fig_5e"
out_dir <- "../../../data/rna/gsva/results/gsva_results"
dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)

tp_order <- c("PreTx", "EI", "ASCT90d", "ASCT1y", "ASCT2y", "Healthy")
stat_comparisons <- list(c("Healthy", "PreTx"), c("Healthy", "EI"), c("Healthy", "ASCT90d"), c("Healthy", "ASCT1y"), c("Healthy", "ASCT2y"))

tissue <- "pbmc" 
gsva_all <- fread(file.path(out_dir, sprintf("gsva_scores_%s.csv", tissue)))
stats_all <- fread(file.path(out_dir, sprintf("gsva_stats_%s_adjusted.csv", tissue)))

celltypes_to_plot <- c(
  "CD4 T Mem KLRF1- GZMB+",
  "CD4 T Mem ISG+",
  "CD4 T EM GZMB- CD27-",
  "CD4 T EM GZMB- CD27+",
  "CD4 T CM",
  "gdT Vd2 GZMK+",
  "gdT Vd2 GZMB+",
  "gdT Vd1 Eff KLRF1+",
  "Prolif T",
  "Prolif NK",
  "DN T",
  "CD8 T Mem ISG+",
  "CD8 T EM KLRF1- GZMB+",
  "CD8 T EM KLRF1+ GZMB+",
  "CD8 T EM GZMK- CD27+",
  "CD8 T EM GZMK+ CD27+",
  "CD8 T CM",
  "NK Adaptive",
  "CD56dim NK ISG+",
  "CD56dim NK GZMK-",
  "CD56dim NK GZMK+",
  "CD56br NK"
)

pathways_to_plot <- match_pathways(c(
  "EPSTEIN-BARR VIRUS INFECTION",
  "HYPOXIA",
  "INTERFERON ALPHA RESPONSE",
  "INTERFERON GAMMA RESPONSE",
  "OXIDATIVE PHOSPHORYLATION",
  "TCR SIGNALING",
  "TGF-BETA SIGNALING",
  "TNF-ALPHA SIGNALING VIA NF-KB"
), gsva_all)

In [14]:
for (ct in celltypes_to_plot) {
  plot_gsva_violins(ct, pathways_to_plot, tissue)
}

Saved: results/gsva_plots/fig_5e/gsva_pbmc_CD4_T_Mem_KLRF1-_GZMB+.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_CD4_T_Mem_ISG+.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_CD4_T_EM_GZMB-_CD27-.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_CD4_T_EM_GZMB-_CD27+.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_CD4_T_CM.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_gdT_Vd2_GZMK+.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_gdT_Vd2_GZMB+.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_gdT_Vd1_Eff_KLRF1+.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_Prolif_T.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_Prolif_NK.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_DN_T.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_CD8_T_Mem_ISG+.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_CD8_T_EM_KLRF1-_GZMB+.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_CD8_T_EM_KLRF1+_GZMB+.pdf

Saved: results/gsva_plots/fig_5e/gsva_pbmc_CD8_T_EM_GZMK-_CD27+.pdf

Saved: results/gsva_plots/fig_5e/gsva_pb

## Revision Figure 6F - PBMC

In [15]:
# ===== Figure 6F — ASCT1y vs. Healthy (PBMC B cells + DCs) =====
tissue <- "pbmc"
out_dir <- "../../../data/rna/gsva/results/gsva_results"
plot_dir <- "../../../data/rna/gsva/results/gsva_plots/fig_6f"
dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)

tp_order <- c("ASCT1y", "Healthy")
stat_comparisons <- list(c("Healthy", "ASCT1y"))

# data + FGSEA-matched stats for this tissue
gsva_all <- data.table::fread(file.path(out_dir, sprintf("gsva_scores_%s.csv", tissue)))
stats_all <- data.table::fread(file.path(out_dir, sprintf(
  "gsva_stats_%s_adjusted.csv",
  tissue
)))

celltypes_to_plot <- c(
  "Trans B", "Eff B CD27-", "Eff B CD27+",
  "Naive B Core", "Mem B Early", "Mem B Core", 
    "Mem B CD95", "cDC2 HLA-DRhi", "cDC2 CD14+", "cDC2 ISG+", "cDC1", "pDC"
)

pathways_to_plot <- c(
  # BCR signaling / NF-kB (PLCG2 claim)
  "REACTOME_SIGNALING BY THE B CELL RECEPTOR (BCR)",
  "REACTOME_ANTIGEN ACTIVATES B CELL RECEPTOR (BCR) LEADING TO GENERATION OF SECOND MESSENGERS",
  "REACTOME_DOWNSTREAM SIGNALING EVENTS OF B CELL RECEPTOR (BCR)",
  "REACTOME_PLC BETA MEDIATED EVENTS",
  "REACTOME_DAG AND IP3 SIGNALING",
  "REACTOME_ACTIVATION OF NF-KAPPAB IN B CELLS",
  "KEGG_B CELL RECEPTOR SIGNALING PATHWAY",
  "KEGG_CALCIUM SIGNALING PATHWAY",
  "ENCODE_RELA ENCODE",
  "HALLMARK_TNF-ALPHA SIGNALING VIA NF-KB",
  # Group B — Antigen processing / MHC-I / IFN (HLA-C, PSMB8 claim)
  "REACTOME_CLASS I MHC MEDIATED ANTIGEN PROCESSING & PRESENTATION",
  "REACTOME_ANTIGEN PRESENTATION FOLDING, ASSEMBLY AND PEPTIDE LOADING OF CLASS I MHC",
  "REACTOME_ER-PHAGOSOME PATHWAY",
  "REACTOME_ANTIGEN PROCESSING UBIQUITINATION & PROTEASOME DEGRADATION",
  "REACTOME_INTERFERON GAMMA SIGNALING",
  "REACTOME_INTERFERON ALPHA BETA SIGNALING",
  "KEGG_ANTIGEN PROCESSING AND PRESENTATION",
  "KEGG_PROTEASOME",
  "ENCODE_IRF1 ENCODE",
  "HALLMARK_INTERFERON GAMMA RESPONSE",
  "HALLMARK_INTERFERON ALPHA RESPONSE",
  # Group C — Virus (figure 6F)
  "KEGG_INFLUENZA A",
  "KEGG_EPSTEIN-BARR VIRUS INFECTION"
)

# resolve against the data; report anything missing
miss_pw <- setdiff(pathways_to_plot, unique(gsva_all$pathway))
miss_ct <- setdiff(celltypes_to_plot, unique(gsva_all$celltype))
if (length(miss_pw)) message("PATHWAYS NOT FOUND:\n  ", paste(miss_pw, collapse = "\n  "))
if (length(miss_ct)) message("CELL TYPES NOT FOUND:\n  ", paste(miss_ct, collapse = "\n  "))

pathways_to_plot <- intersect(pathways_to_plot, unique(gsva_all$pathway))
celltypes_to_plot <- intersect(celltypes_to_plot, unique(gsva_all$celltype))

In [16]:
for (ct in celltypes_to_plot) plot_gsva_violins(ct, pathways_to_plot, tissue)

Saved: results/gsva_plots/fig_6f/gsva_pbmc_Trans_B.pdf

Saved: results/gsva_plots/fig_6f/gsva_pbmc_Eff_B_CD27-.pdf

Saved: results/gsva_plots/fig_6f/gsva_pbmc_Eff_B_CD27+.pdf

Saved: results/gsva_plots/fig_6f/gsva_pbmc_Naive_B_Core.pdf

Saved: results/gsva_plots/fig_6f/gsva_pbmc_Mem_B_Early.pdf

Saved: results/gsva_plots/fig_6f/gsva_pbmc_Mem_B_Core.pdf

Saved: results/gsva_plots/fig_6f/gsva_pbmc_Mem_B_CD95.pdf

Saved: results/gsva_plots/fig_6f/gsva_pbmc_cDC2_HLA-DRhi.pdf

Saved: results/gsva_plots/fig_6f/gsva_pbmc_cDC2_CD14+.pdf

Saved: results/gsva_plots/fig_6f/gsva_pbmc_cDC2_ISG+.pdf

Saved: results/gsva_plots/fig_6f/gsva_pbmc_cDC1.pdf

Saved: results/gsva_plots/fig_6f/gsva_pbmc_pDC.pdf

